In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import shutil
import json
from pathlib import Path

import numpy as np

from phd_project.scripts.templates.copy_templates_to_folders import (
    # copy_pushover_config,
    copy_analysis_config,
    copy_structural_model,
    configure_batch_run_file,
    copy_file,
)
from phd_project.scripts.case_study_design_scripts.design_file_helpers import (
    get_control_node_from_design_file,
    get_n_primary_modes_from_design_file,
    get_n_damping_modes_from_design_file,
    remove_soft_storey_braces,
)

from phd_project.config import config

cfg = config.load_config()

In [ ]:
from phd_project.scripts.loading_protocols import (
    FEMA_461_loading_protocol,
    get_FEMA461_displacements_for_building,
)

In [32]:
# INPUTS
design_file_root = cfg["models"]["casestudy_designs_ec8_gen2"]
analysis_root_folder = cfg["analysis_data"]["dc2_sdof_fitting"]

po_batch_run_filename = "pushover_analyses"
modal_batch_run_filename = "modal_analyses"

# PO Parameters
max_drift = 6       # in %; roof drift limit for the pushover analysis.
drift_step = 0.005   # in %; the step size for the pushover analysis.
remove_recorders = False  # if True brace elements deleted if the gussets fail

# CPO Parameters
disp_step = 1.0     # mm
U_max = 250
n_levels = 12
# NLTHA Parameters
gm_json_src_str = 'E:/gm_records_p695'

In [33]:
building_tags = [f.name for f in design_file_root.iterdir() 
                 if f.is_dir() and (f / f"{f.name}_out.json").exists()]

# building_tags = ["3s_cbf_dc2_41"]

In [ ]:
po_scripts_configs_list_dc2 = []
cpo_scripts_configs_list = []
modal_scripts_configs_list_dc2 = []
nltha_scripts_configs_list_dc2 = []

for building_tag in building_tags:

    if building_tag.split("_")[2] != "dc2":
        continue
    
    # create the folder and files for each building model
    building_analysis_folder = analysis_root_folder / building_tag
    building_analysis_folder.mkdir(parents=True, exist_ok=True)

    # copy in the design file for the CBF structure
    building_design_file_src = design_file_root / f"{building_tag}/{building_tag}_out.json"
    building_design_file_dst = building_analysis_folder / f"{building_tag}_designfile.json"

    copy_file(building_design_file_src, building_design_file_dst)

    ###### Structural Model File
    model_src = cfg["templates"]["structural_model"]
    model_dst = building_analysis_folder / "structural_model.py"

    n_damping_modes = get_n_damping_modes_from_design_file(building_design_file_dst)
    damping_updates = {"n_modes": n_damping_modes}
    recorder_updates = {"incl_remove_recorders": remove_recorders}

    copy_structural_model(
        model_src, 
        model_dst, 
        design_json=building_design_file_dst.name,
        recorder_updates=recorder_updates,
        damping_updates=damping_updates,
        )

    ###### Pushover Files
    # copy the run po analysis file
    po_analysis_src = cfg["templates"]["run_pushover"]
    po_analysis_dst = building_analysis_folder / "run_pushover.py"

    copy_file(po_analysis_src, po_analysis_dst)

    # create the po analysis config file
    po_config_src = cfg["templates"]["config_pushover"]
    po_config_dst = building_analysis_folder / "config_pushover.py"

    ctrl_node = get_control_node_from_design_file(building_design_file_dst)
    disp_type = "drift"
    Umax = max_drift
    dU = drift_step

    po_outfolder = f"pushover"
    config_updates = {"displacement_type": disp_type, "U_max": Umax, "dU": dU, "ctrl_node": ctrl_node}

    copy_analysis_config(po_config_src, po_config_dst, 
                         results_folder_name=po_outfolder, update_config=config_updates)


    # add to scripts and configs list for batch file run
    po_scripts_configs_list_dc2.append({
        "script": po_analysis_dst,
        "config": [po_config_dst]
        })

    ###### Cyclic Pushover Files
    # copy the run po analysis file
    cpo_analysis_src = cfg["templates"]["run_cyclic_pushover"]
    cpo_analysis_dst = building_analysis_folder / "run_cyclic_pushover.py"

    copy_file(cpo_analysis_src, cpo_analysis_dst)

    # create the po analysis config file
    cpo_config_src = cfg["templates"]["config_cyclic_pushover"]
    cpo_config_dst = building_analysis_folder / "config_cyclic_pushover.py"

    ctrl_node = get_control_node_from_design_file(building_design_file_dst)
    disp_type = "displacement"
    dU = disp_step

    n_storeys = int(building_tag[0])
    displacements = np.round(get_FEMA461_displacements_for_building(building_design_file_dst, n_levels, U_max), 3).tolist()
    cpo_outfolder = f"cyclic_pushover"
    config_updates = {"displacement_type": disp_type, "dU": dU, "ctrl_node": ctrl_node, "displacements": displacements}

    copy_analysis_config(cpo_config_src, cpo_config_dst, 
                         results_folder_name=cpo_outfolder, update_config=config_updates)

    # add to scripts and configs list for batch file run
    cpo_scripts_configs_list.append({
        "script": cpo_analysis_dst,
        "config": [cpo_config_dst]
        })


    ###### NLTHA Files
    # # copy the run nltha analysis file
    # nltha_analysis_src = cfg["templates"]["run_nltha_ud"]
    # nltha_analysis_dst = building_analysis_folder / "run_nltha_ud.py"

    # copy_file(nltha_analysis_src, nltha_analysis_dst)
    # 
    # create the nltha config file
    # nltha_config_src = cfg["templates"]["config_nltha"]
    # for record, sf in [
    #     ("fema_p695_120121.json", 2.0), 
    #     ("fema_p695_120621.json", 2.6)]: 
        
    #     rec_num = record.split(".json")[0].split("_")[-1]

    #     nltha_config_dst = building_analysis_folder / f"config_nltha_{rec_num}.py"

    #     nltha_outfolder = f"nltha_{rec_num}_sf{int(sf*1000)}"
    #     config_updates = {
    #         'gm_json_file': record,
    #         'scale_factor': sf
    #     }

    #     copy_analysis_config(nltha_config_src, nltha_config_dst, 
    #                          results_folder_name=nltha_outfolder, 
    #                          update_config=config_updates,
    #                          gm_json_src_str=gm_json_src_str)
        
    #     # add to scripts and configs list for batch file run
    #     nltha_scripts_configs_list_dc2.append({
    #         "script": nltha_analysis_dst,
    #         "config": [nltha_config_dst]
    #         })

    ###### Modal Analysis Files
    # copy the run modal analysis file
    modal_analysis_src = cfg["templates"]["run_modal"]
    modal_analysis_dst = building_analysis_folder / "run_modal.py"

    copy_file(modal_analysis_src, modal_analysis_dst)

    # create the modal analysis config file
    modal_config_src = cfg["templates"]["config_modal"]
    modal_config_dst = building_analysis_folder / "config_modal.py"

    ctrl_node = get_control_node_from_design_file(building_design_file_dst)
    Umax = ("drift", max_drift)
    dU = drift_step

    modal_outfolder = f"modal"
    config_updates = {
        "n_modes": get_n_primary_modes_from_design_file(building_design_file_dst)
    }

    copy_analysis_config(modal_config_src, modal_config_dst, 
                         results_folder_name=modal_outfolder, 
                         update_config=config_updates)

    # add to scripts and configs list for batch file run
    modal_scripts_configs_list_dc2.append({
        "script": modal_analysis_dst,
        "config": [modal_config_dst]
        })

In [35]:
# create a script for batch running the analyses
batch_run_src = cfg["templates"]["batch_run"]

# Pushover batch files
for dc, scripts_configs_list in [
    ("dc2", po_scripts_configs_list_dc2), 
    ]:

    batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / f"mdof_po.py"
    configure_batch_run_file(batch_run_src, batch_run_dst, scripts_configs_list)

# Cyclic Pushover batch files
for dc, scripts_configs_list in [
    ("dc2", cpo_scripts_configs_list), 
    ]:

    batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / f"mdof_cpo.py"
    configure_batch_run_file(batch_run_src, batch_run_dst, scripts_configs_list)

# Modal batch files
for dc, scripts_configs_list in [
    ("dc2", modal_scripts_configs_list_dc2), 
    ]:

    batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / f"mdof_modal.py"
    configure_batch_run_file(batch_run_src, batch_run_dst, scripts_configs_list)

# NLTHA batch files
# for dc, scripts_configs_list in [
#     ("dc2", nltha_scripts_configs_list_dc2), 
#     ]:

#     batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / f"{nltha_batch_run_filename}_{dc}.py"
#     configure_batch_run_file(batch_run_src, batch_run_dst, scripts_configs_list)

In [ ]:
def clone_and_rename_folder(src_path, dst_path, exclude_list, print_successes:bool=False):
    """
    Copies directories in root_path, appends '_ss' to names,
    and excludes specified files/folders.
    """
    # Convert list to a format shutil.copytree understands
    ignore_func = shutil.ignore_patterns(*exclude_list)
           
    # Only process directories that don't already end in _ss
    if os.path.isdir(src_path):       
        
        # If the destination exists, remove it entirely to "replace" it
        if os.path.exists(dst_path):
            if print_successes:
                print(f"Replacing existing folder: {dst_path}")
            shutil.rmtree(dst_path)

        try:
            shutil.copytree(src_path, dst_path, ignore=ignore_func)
            if print_successes:
                print(f"Successfully copied: {src_path} -> {dst_path}")
        except Exception as e:
            print(f"Error copying {src_path}: {e}")

In [ ]:
# Copy folders and the files inside
building_folders = [analysis_root_folder / bf for bf in os.listdir(analysis_root_folder) 
                    if len(bf.split("_")) == 4]
exclude = ["pushover", "config_nltha_120121.py", "config_nltha_120621.py", "run_nltha_ud.py", "cyclic_pushover", "modal"]

batch_modal = []
batch_po = []
batch_cpo = []
for folder in building_folders:
    new_folder = folder.parent / f"{folder.name}_ss"
    clone_and_rename_folder(folder, new_folder, exclude)

    old_filename = f"{folder.name}_designfile.json"
    new_filename = f"{folder.name}_ss_designfile.json"
    remove_soft_storey_braces(new_folder / old_filename, new_folder / new_filename, [(1, 1), (3, 1)])
    os.remove(new_folder / old_filename)

    # update the structural model file
    copy_structural_model(new_folder / "structural_model.py", new_folder / "structural_model.py", design_json=new_filename)

    batch_modal.append({
        "script": new_folder / "run_modal.py",
        "config": [new_folder / "config_modal.py"]
        })
    
    batch_po.append({
        "script": new_folder / "run_pushover.py",
        "config": [new_folder / "config_pushover.py"]
        })
    
    batch_cpo.append({
        "script": new_folder / "run_cyclic_pushover.py",
        "config": [new_folder / "config_cyclic_pushover.py"]
        })
    
# create batch file
batch_run_src = cfg["templates"]["batch_run"]
batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / "mdof_ss_modal.py"
configure_batch_run_file(batch_run_src, batch_run_dst, batch_modal)

batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / "mdof_ss_po.py"
configure_batch_run_file(batch_run_src, batch_run_dst, batch_po)

batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / "mdof_ss_cpo.py"
configure_batch_run_file(batch_run_src, batch_run_dst, batch_cpo)